In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("train_FD001.txt", sep=r"\s+", header=None)
df.columns=['Unit', 'Cycles', 'Ops1', 'Ops2', 'Ops3', 'Sensor1', 'Sensor2', 'Sensor3', 'Sensor4', 'Sensor5', 'Sensor6', 'Sensor7', 'Sensor8', 'Sensor9', 'Sensor10', 'Sensor11', 'Sensor12', 'Sensor13', 'Sensor14', 'Sensor15', 'Sensor16', 'Sensor17', 'Sensor18', 'Sensor19', 'Sensor20', 'Sensor21']

In [3]:
df = df.dropna(axis=1, how="all")

In [4]:
df.shape

(20631, 26)

In [5]:
df = df.apply(pd.to_numeric, errors="coerce")

In [6]:
df.isnull().sum()


,0
Unit,0
Cycles,0
Ops1,0
Ops2,0
Ops3,0
Sensor1,0
Sensor2,0
Sensor3,0
Sensor4,0
Sensor5,0


In [7]:
df = df.drop_duplicates()


In [8]:
df.nunique()


,0
Unit,100
Cycles,362
Ops1,158
Ops2,13
Ops3,1
Sensor1,1
Sensor2,310
Sensor3,3012
Sensor4,4051
Sensor5,1


In [9]:
df = df.drop(columns=['Ops3', 'Sensor1', 'Sensor5', 'Sensor6', 'Sensor10', 'Sensor16', 'Sensor18', 'Sensor19'])

In [10]:
max_cycles = df.groupby("Unit")["Cycles"].max()


In [11]:
df['RUL'] = df['Unit'].map(max_cycles) - df['Cycles']

Clean Test Data

In [12]:
df_test = pd.read_csv("test_FD001.txt", sep=r"\s+", header=None)
df_test.columns=['Unit', 'Cycles', 'Ops1', 'Ops2', 'Ops3', 'Sensor1', 'Sensor2', 'Sensor3', 'Sensor4', 'Sensor5', 'Sensor6', 'Sensor7', 'Sensor8', 'Sensor9', 'Sensor10', 'Sensor11', 'Sensor12', 'Sensor13', 'Sensor14', 'Sensor15', 'Sensor16', 'Sensor17', 'Sensor18', 'Sensor19', 'Sensor20', 'Sensor21']

In [13]:
df_test = df_test.dropna(axis=1, how="all")

In [14]:
df_test.shape

(13096, 26)

In [15]:
df_test = df_test.apply(pd.to_numeric, errors="coerce")

In [16]:
df_test.isnull().sum()


,0
Unit,0
Cycles,0
Ops1,0
Ops2,0
Ops3,0
Sensor1,0
Sensor2,0
Sensor3,0
Sensor4,0
Sensor5,0


In [17]:
df_test.nunique()


,0
Unit,100
Cycles,303
Ops1,150
Ops2,14
Ops3,1
Sensor1,1
Sensor2,262
Sensor3,2361
Sensor4,2954
Sensor5,1


In [18]:
df_test = df_test.drop(columns=['Ops3', 'Sensor1', 'Sensor5', 'Sensor6', 'Sensor10', 'Sensor16', 'Sensor18', 'Sensor19'])

Train Model and Test Using Test Data


In [19]:
df["life_used_pct"] = df["Cycles"] / df["Cycles"].max()

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [21]:
from xgboost import XGBRegressor

In [22]:
rul_df = pd.read_csv(
    "RUL_FD001.txt",
    header=None
)

rul_df.columns = ["final_RUL"]

In [23]:
test_max = df_test.groupby("Unit")["Cycles"].max().reset_index()
test_max.columns = ["Unit", "max_cycle"]

test_max["final_RUL"] = rul_df["final_RUL"]

test_df = df_test.merge(test_max, on="Unit")

test_df["RUL"] = (
    test_df["max_cycle"] + test_df["final_RUL"] - test_df["Cycles"]
)
test_df["life_used_pct"] = test_df["Cycles"] / test_df["Cycles"].max()

In [24]:
drop_cols = ["Unit", "max_cycle", "final_RUL", "RUL"]

X_train = df.drop(columns=["Unit", "RUL"])
y_train = df["RUL"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["RUL"]

In [25]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [26]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [27]:
predictions = model.predict(X_test)

In [28]:
import numpy as np

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\nMODEL RESULTS USING test_FD001")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))


MODEL RESULTS USING test_FD001
MAE : 31.44
RMSE: 41.67
R²  : 0.501


In [29]:
results = pd.DataFrame({
    "Actual_RUL": y_test.values,
    "Predicted_RUL": predictions
})


In [34]:
results["error"] = results["Actual_RUL"] - results["Predicted_RUL"]

results["risk_level"] = np.where(
    results["Predicted_RUL"] <= 30, "High",
    np.where(results["Predicted_RUL"] <= 60, "Medium", "Low")
)

In [40]:
powerbi_data = test_df.copy()

# Add RUL related columns from results
powerbi_data["predicted_RUL"] = results["Predicted_RUL"]
powerbi_data["error"] = results["error"]
powerbi_data["risk_level"] = results["risk_level"]

# Define a dictionary for column renaming
column_rename_map = {
    "Unit": "unit_number",
    "Cycles": "cycle",
    "Ops1": "op1",
    "Ops2": "op2"
}

# Rename existing columns
powerbi_data = powerbi_data.rename(columns=column_rename_map)

# Prepare the final list of columns for PowerBI
final_columns = [
    "unit_number",
    "cycle",
    "op1",
    "op2",
    "predicted_RUL",
    "RUL", # This RUL is from test_df
    "error",
    "risk_level"
]

# Add existing sensor columns with the desired naming convention
for i in range(1, 22):
    df_sensor_name = f"Sensor{i}"
    output_sensor_name = f"sensor_{i}"
    if df_sensor_name in powerbi_data.columns:
        final_columns.append(df_sensor_name)
        column_rename_map[df_sensor_name] = output_sensor_name

# Select and rename the final columns
powerbi_data = powerbi_data[final_columns].rename(columns=column_rename_map)

powerbi_data.to_csv("powerbi_dataset.csv", index=False)

print("Saved: powerbi_dataset.csv")

Saved: powerbi_dataset.csv


In [36]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTOP 10 FEATURES")
print(importance.head(10))


TOP 10 FEATURES
          Feature  Importance
17  life_used_pct    0.388319
0          Cycles    0.262010
9        Sensor11    0.099796
5         Sensor4    0.041485
10       Sensor12    0.032460
8         Sensor9    0.026323
6         Sensor7    0.020306
13       Sensor15    0.015731
7         Sensor8    0.014980
12       Sensor14    0.014954
